# 📘 Module 4.2 — Vision Transformer (ViT)

**Goal:** Understand how Transformers are applied to images — the foundation of modern vision in ADAS.

## ViT: Transforming Computer Vision
The key insight of ViT (Dosovitskiy et al., 2020): **An image is a sequence of patches**, just like a sentence is a sequence of words!

```
Image (224×224)  →  Split into 16×16 patches  →  196 "tokens"  →  Transformer
```

---

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

## 1. Patch Embedding — Images as Sequences

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and project to embeddings."""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Conv2d with kernel_size=patch_size effectively splits and projects
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        # x: (batch, 3, 224, 224)
        x = self.projection(x)    # (batch, embed_dim, 14, 14)
        x = x.flatten(2)          # (batch, embed_dim, 196)
        x = x.transpose(1, 2)     # (batch, 196, embed_dim) — sequence format!
        return x

# Test
patch_embed = PatchEmbedding()
img = torch.randn(1, 3, 224, 224)
patches = patch_embed(img)
print(f"Image: {img.shape}")
print(f"Patches: {patches.shape}")
print(f"Number of patches: {patch_embed.num_patches} ({224//16}×{224//16})")
print(f"Each patch is projected to {patches.shape[-1]} dimensions")

In [ ]:
# --- Visualize how an image is split into patches ---
img_np = np.random.rand(224, 224, 3) * 0.5 + 0.25  # Synthetic image

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
axes[0].imshow(img_np)
axes[0].set_title('Original Image (224×224)')
axes[0].axis('off')

# Image with patch grid
axes[1].imshow(img_np)
patch_size = 16
for i in range(0, 224, patch_size):
    axes[1].axhline(y=i, color='red', linewidth=0.5)
    axes[1].axvline(x=i, color='red', linewidth=0.5)
axes[1].set_title(f'Split into {(224//patch_size)**2} patches ({patch_size}×{patch_size})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. Complete Vision Transformer

```
Image → Patch Embedding → [CLS] + Patches + Position Encoding
                                      │
                              Transformer Encoder (×L layers)
                                      │
                              [CLS] token → Classification Head
```

In [ ]:
class VisionTransformer(nn.Module):
    """
    Vision Transformer (ViT) — simplified implementation.
    Based on "An Image is Worth 16x16 Words" (Dosovitskiy et al., 2020)
    """
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, embed_dim=768, depth=12,
                 num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        self.num_patches = (img_size // patch_size) ** 2
        
        # 1. Patch Embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        
        # 2. CLS token (learnable classification token)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
        # 3. Position Embedding (learnable)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        
        # 4. Transformer Encoder blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        
        # 5. Classification Head
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)
        
        # Prepend CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, num_patches+1, embed_dim)
        
        # Add positional encoding
        x = self.pos_drop(x + self.pos_embed)
        
        # Transformer encoder
        x = self.transformer(x)
        
        # Classification: use CLS token output
        x = self.norm(x[:, 0])  # Take CLS token
        x = self.head(x)
        
        return x

# Create a small ViT for demonstration
vit = VisionTransformer(
    img_size=224, patch_size=16,
    num_classes=10,   # ADAS: car, person, truck, etc.
    embed_dim=256,    # Smaller for demo (ViT-Base uses 768)
    depth=6,          # 6 layers (ViT-Base uses 12)
    num_heads=8,
)

x = torch.randn(2, 3, 224, 224)
out = vit(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"\nTotal parameters: {sum(p.numel() for p in vit.parameters()):,}")

## 3. ViT Variants & ADAS Applications

| Model | Params | Resolution | ADAS Usage |
|-------|--------|------------|------------|
| ViT-Tiny | 5.7M | 224 | Edge deployment |
| ViT-Small | 22M | 224 | Embedded systems |
| ViT-Base | 86M | 224/384 | Standard backbone |
| ViT-Large | 307M | 224/384 | High-accuracy |
| DeiT | 22M | 224 | Data-efficient training |
| Swin Transformer | 29-197M | 224+ | Hierarchical features |

In [ ]:
# --- Using Pretrained ViT from torchvision ---
import torchvision.models as models

# Available ViT models (uncomment to download)
# vit_b_16 = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)

# For demo: without pretrained weights
vit_b_16 = models.vit_b_16(weights=None)

print(f"ViT-Base/16 parameters: {sum(p.numel() for p in vit_b_16.parameters()):,}")

# Adapt for ADAS classification
vit_b_16.heads = nn.Linear(768, 5)  # 5 ADAS classes
x = torch.randn(1, 3, 224, 224)
print(f"Input: {x.shape} → Output: {vit_b_16(x).shape}")

---
## ✅ Key Takeaways

1. **ViT** treats images as sequences of patches — bringing NLP-like transformers to vision
2. **Patch embedding** converts image patches to token embeddings
3. **CLS token** aggregates information from all patches for classification
4. ViTs need **large datasets** or **pretraining** (ImageNet-21k) to work well
5. **Swin Transformer** adds hierarchical features, making it better for detection/segmentation

---
## 📖 Next Steps
➡️ **Next notebook:** [03_detr_object_detection.ipynb](03_detr_object_detection.ipynb) — DETR: Transformer-based object detection